# Phase 3 Lab - Async Jobs

Mục tiêu: hiểu vì sao backend trả `job_id` ngay và worker xử lý nền.

Expected output chính: worker tests pass và lifecycle logic cover `pending`,
`running`, `completed`, `failed`, stale-running recovery.

Safety: mock-only, không network, không model calls.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy worker tests

Command này là verification chính cho Phase 3.


In [ ]:
run(["uv", "run", "pytest", "tests/test_worker.py", "-q", "--tb=short"], timeout=180)


## 2. Xem status lifecycle từ schema/test names

Cell này không xử lý DB. Nó giúp bạn map lifecycle mà tests đang bảo vệ.


In [ ]:
lifecycle = [
    "pending -> running -> completed",
    "pending -> running -> failed",
    "completed -> skipped when called again",
    "failed -> skipped when called again",
    "fresh running -> skipped",
    "stale running -> recovered",
]

for item in lifecycle:
    print(item)


## 3. Manual flow nếu muốn mở server

Không chạy server trong notebook để tránh cell treo. Chạy trong terminal:

```bash
cd shopping_assistant_v3
uv run uvicorn backend.api.main:app --host 127.0.0.1 --port 8000
```

Terminal khác:

```bash
curl -X POST http://127.0.0.1:8000/api/chat-jobs \
  -H "Content-Type: application/json" \
  -d '{"message":"test"}'
curl http://127.0.0.1:8000/api/chat-jobs/<job_id>
```


## 4. Cách đọc kết quả

- `tests/test_worker.py` pass: worker lifecycle đúng.
- Manual `POST` trả `job_id` ngay.
- Manual `GET` có thể cần poll vài lần trước khi completed.
- Current Phase 5A code trả result giàu hơn historical Phase 3 mock result.
